# 160 — Prompt injection e instrucciones no confiables

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El laboratorio `safety` ejercita el registro y veredicto de comportamiento ante
entradas potencialmente maliciosas: el contrato JSON (kind + evidence) es lo que una barrera de
validación inspeccionaría antes de permitir una acción.


In [ ]:
result = run_lab("safety", seed=160)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


**Ejercicio 2.**

```text
(a) DIRECTA   — superficie: mensaje del usuario
(b) INDIRECTA — superficie: contenido recuperado (PDF); texto oculto
(c) INDIRECTA — superficie: dato externo (reseña) que el agente procesa
(d) DIRECTA   — superficie: mensaje del usuario (persona-jailbreak)
```

**Ejercicio 3.** Orden de daño creciente: (i) chatbot sin herramientas < (ii) lectura sin envío
< (iii) `send_email` arbitrario. Regla: **el radio de daño de una inyección exitosa es igual a los
permisos/capacidades del modelo**. Sin acción, el peor caso es texto malo; con exfiltración por
envío arbitrario, el peor caso es fuga de datos a un tercero.

**Ejercicio 4 (esquema).** (1) *Privilegio mínimo*: `issue_refund` limitado a un monto máximo y a
la cuenta del ticket — si es la única capa, no evita reembolsos indebidos dentro del límite.
(2) *Separación de confianza*: el texto del ticket entra marcado como no confiable — si es la
única, el modelo puede ignorar el marcado. (3) *Validación de salida*: todo reembolso > umbral
requiere aprobación humana — si es la única, no cubre montos bajos masivos. (4) *Detección*:
clasificador de instrucciones incrustadas en tickets — si es la única, es evadible y genera falsos
positivos. La seguridad emerge de combinarlas, no de ninguna aislada.


In [ ]:
# Verificación del Ejercicio 2
casos = {
    "a": ("directa", "mensaje del usuario"),
    "b": ("indirecta", "PDF recuperado"),
    "c": ("indirecta", "resena externa"),
    "d": ("directa", "mensaje del usuario"),
}
directas = [k for k, (t, _) in casos.items() if t == "directa"]
indirectas = [k for k, (t, _) in casos.items() if t == "indirecta"]
print("directas:", directas, "| indirectas:", indirectas)
assert directas == ["a", "d"] and indirectas == ["b", "c"]


## Reflexión (guía)

1. Porque el LLM no tiene una frontera fiable entre "instrucción con autoridad" y "dato a
   procesar"; ambos son texto concatenado en la misma ventana de contexto.
2. El **privilegio mínimo** y la **validación/aprobación determinista** de acciones: acotan o
   bloquean el daño aunque el modelo obedezca la inyección, sin confiar en su comportamiento.
3. Con una suite de regresión adversarial (clases 158-159): los casos de inyección conocidos se
   re-ejecutan en cada cambio de modelo/prompt y el despliegue se bloquea si reaparecen.
